<a href="https://colab.research.google.com/github/verdantrays/ml-zoomcamp-datatalks-club/blob/main/homework_week_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import torch
torch.__version__

'2.9.0+cu126'

In [2]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!unzip -q data.zip

--2025-12-09 02:23:19--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-09T02%3A56%3A28Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-09T01%3A55%3A39Z&ske=2025-12-09T02%3A56%3A28Z&sks=b&skv=2018-11-09&sig=vzWrxkNOOVpm8PzBvPWt115ubvcWiuCkFzTO7n55d50%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NTI0ODc5OSwibmJmIjoxNzY1MjQ2OTk5LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG

In [12]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [13]:
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [14]:
train_transforms = transforms.Compose([
    transforms.Resize((200,200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
test_transforms = train_transforms

In [15]:
train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
validation_dataset = datasets.ImageFolder("data/test", transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=20, shuffle=False)

In [16]:
class HairCNN(nn.Module):
    def __init__(self):
        super(HairCNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.relu2 = nn.ReLU()
        self.fc2 = nn.Linear(64,1)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.relu2(self.fc1(x))
        return self.fc2(x)

In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = HairCNN().to(device)

In [18]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

In [19]:
total_params = sum(p.numel() for p in model.parameters())
total_params

20073473

In [20]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(epoch+1, epoch_loss, epoch_acc, val_epoch_loss, val_epoch_acc)

1 0.6462258949875832 0.63625 0.6032439594838157 0.6517412935323383
2 0.547510139644146 0.71 0.7250668498043397 0.6318407960199005
3 0.5532774537801742 0.725 0.5990864308319281 0.6716417910447762
4 0.4802091151475906 0.77125 0.6032694945881023 0.6567164179104478
5 0.4333878390491009 0.8025 0.6196025607004687 0.6766169154228856
6 0.374002392962575 0.8325 0.7371423721906558 0.6766169154228856
7 0.27212329134345054 0.88375 0.9222679344279257 0.6417910447761194
8 0.247810354270041 0.9 0.729401338456282 0.7213930348258707
9 0.20747391935437917 0.92 0.7522510044017241 0.7014925373134329
10 0.14940188731998205 0.945 0.7893380739202547 0.7014925373134329


In [21]:
np.median(history['acc'])

np.float64(0.8175)

In [22]:
np.std(history['loss'])

np.float64(0.1589653363477039)

In [23]:
train_transforms = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9,1.0), ratio=(0.9,1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [24]:
train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

In [33]:
for epoch in range(10, 20):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val

    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/20 - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 11/20 - Loss: 0.3968, Acc: 0.8175, Val Loss: 0.5119, Val Acc: 0.7662
Epoch 12/20 - Loss: 0.4159, Acc: 0.8063, Val Loss: 0.5232, Val Acc: 0.7562
Epoch 13/20 - Loss: 0.3740, Acc: 0.8325, Val Loss: 0.5415, Val Acc: 0.7512
Epoch 14/20 - Loss: 0.3845, Acc: 0.8250, Val Loss: 0.5837, Val Acc: 0.7413
Epoch 15/20 - Loss: 0.4244, Acc: 0.8137, Val Loss: 0.5227, Val Acc: 0.7662
Epoch 16/20 - Loss: 0.3954, Acc: 0.8313, Val Loss: 0.5055, Val Acc: 0.8010
Epoch 17/20 - Loss: 0.3713, Acc: 0.8363, Val Loss: 0.5618, Val Acc: 0.7662
Epoch 18/20 - Loss: 0.3552, Acc: 0.8275, Val Loss: 0.6039, Val Acc: 0.7214
Epoch 19/20 - Loss: 0.3679, Acc: 0.8413, Val Loss: 0.5163, Val Acc: 0.7711
Epoch 20/20 - Loss: 0.3663, Acc: 0.8337, Val Loss: 0.4530, Val Acc: 0.7910


In [40]:
np.mean(history['val_loss'][11:20])

np.float64(0.5346218937335586)

In [44]:
np.mean(history['val_acc'][16:20])

np.float64(0.7624378109452736)